# L'objectif est de creer une IA en machine learning de classification, son but seras donc de verifier si un SMS est un Spam ou non (Spam / ham). Classement binaire.

### DataSet :
* 2 feature [label (spam/ham)/ message]
* 5572 samples => ham = 4825 + spam = 747

### se dataset est donc compose de 5572 exemple de message en englais, a qui leur est assosier un label correspondant a si le message est un spam ou non.
https://www.kaggle.com/datasets/lazycoder00/sms-spam-collection?

In [2]:
import pandas as pd

# On utilise sep='\t' pour les tabulations
# On ajoute names pour définir les colonnes (label et message)
df = pd.read_csv("SMS_Spam/SMSSpamCollection",
                 sep='\t',
                 names=["label", "message"],
                 encoding="latin-1")

print(df.tail())
df['label'].value_counts()

     label                                            message
5567  spam  This is the 2nd time we have tried 2 contact u...
5568   ham              Will Ã¼ b going to esplanade fr home?
5569   ham  Pity, * was in mood for that. So...any other s...
5570   ham  The guy did some bitching but I acted like i'd...
5571   ham                         Rofl. Its true to its name


label
ham     4825
spam     747
Name: count, dtype: int64

**clean du text** du data set pour enlever les majuscule, les ponctuation, et les espace multiple, affin de le simplifier

In [3]:
import re

def clean_text(text):
    text = text.lower()  # minuscules
    text = re.sub(r'\W', ' ', text)  # enlever ponctuation
    text = re.sub(r'\s+', ' ', text)  # enlever espaces multiples
    return text

df['clean_message'] = df['message'].apply(clean_text)
print(df.tail())

     label                                            message  \
5567  spam  This is the 2nd time we have tried 2 contact u...   
5568   ham              Will Ã¼ b going to esplanade fr home?   
5569   ham  Pity, * was in mood for that. So...any other s...   
5570   ham  The guy did some bitching but I acted like i'd...   
5571   ham                         Rofl. Its true to its name   

                                          clean_message  
5567  this is the 2nd time we have tried 2 contact u...  
5568              will ã¼ b going to esplanade fr home   
5569  pity was in mood for that so any other suggest...  
5570  the guy did some bitching but i acted like i d...  
5571                          rofl its true to its name  


**Vectorisation du text** tranformation du text en nombre ce qui permet de mesurer l’importance des mots dans chaque SMS
=> X = matrice (nb_sms × nb_mots)
Chaque ligne = un SMS
Chaque colonne = un mot

In [42]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(ngram_range=(1,2), max_features=5000)   # TfidfVectorizer(ngram_range=(1,2))
X = vectorizer.fit_transform(df['clean_message'])
y = df['label']
y = y.map({'ham': 0, 'spam': 1})

print(X)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 89439 stored elements and shape (5572, 5000)>
  Coords	Values
  (0, 1570)	0.17805771057076958
  (0, 4327)	0.2769685537065954
  (0, 3198)	0.3074731142399502
  (0, 1028)	0.30443682226088814
  (0, 510)	0.29403364095667955
  (0, 2977)	0.1878779856286487
  (0, 1946)	0.12883220977413534
  (0, 685)	0.3321011213317584
  (0, 1642)	0.21715431449576178
  (0, 4754)	0.2659085127724213
  (0, 2176)	0.3321011213317584
  (0, 889)	0.3321011213317584
  (0, 3991)	0.18725958512741409
  (0, 1621)	0.18429157675895702
  (0, 4523)	0.2196460428440534
  (1, 2911)	0.23857747879824964
  (1, 2187)	0.35829112519120737
  (1, 2119)	0.45951058300477976
  (1, 4676)	0.3787392949355993
  (1, 2975)	0.47964300620241607
  (1, 2914)	0.47964300620241607
  (2, 1946)	0.07307874741376354
  (2, 1428)	0.10453750356423572
  (2, 1263)	0.3262556443894787
  (2, 4730)	0.17268865974166347
  :	:
  (5570, 2241)	0.14429730717740463
  (5570, 3641)	0.166715356032241
  (5570, 1384)	

**separation du data set en 2**. une partie pour les test une partie pour l'entrainement.option stratify permet de conserver la proportion de spam/ham dans les deux ensembles

In [43]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2, #80% → train (apprentissage) 20% → test (évaluation)
    random_state=42,
    stratify = y
)

**Modèle 1** Multinomial Naive Bayes fonctionne bien avec des fréquences de mots. or notre donné sont vectorialiser via TfidfVectorizer()

In [44]:
from sklearn.naive_bayes import MultinomialNB

# création du modèle
model_nb = MultinomialNB(alpha=0.05)

# entraînement
model_nb.fit(X_train, y_train)

,alpha,0.05
,force_alpha,True
,fit_prior,True
,class_prior,None


**Modèle 2** Logistic Regression apprend une combinaison de mots pondérés

Chaque mot a un poids :
* "free" → +3 (spam)
* "win" → +2 (spam)
* "hello" → -1 (ham)

In [49]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

model_lr = LogisticRegression(
    C=100,
    class_weight='balanced',
    max_iter=2000,
    #solver='liblinear'
)
model_lr.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,100
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,None
,solver,'lbfgs'
,max_iter,2000
,multi_class,'deprecated'


**Calculer l’accuracy**

In [50]:
y_pred_nb = model_nb.predict(X_test)
accuracy_nb = accuracy_score(y_test, y_pred_nb)
print("Accuracy Naive Bayes :", accuracy_nb)

y_pred_lr = model_lr.predict(X_test)
accuracy_lr = accuracy_score(y_test, y_pred_lr)
print("Accuracy Logistic Regression :", accuracy_lr)

Accuracy Naive Bayes : 0.9865470852017937
Accuracy Logistic Regression : 0.9865470852017937


**Test important a realiser en plus de l'accuracy sur le **model MultinomialNB** **

In [47]:
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import accuracy_score

print("Confusion Matrix :")
print(confusion_matrix(y_test, y_pred_nb))

print("\nClassification Report :")
print(classification_report(y_test, y_pred_nb))


train_pred = model_nb.predict(X_train)
print("Train accuracy :", accuracy_score(y_train, train_pred))
print("Test accuracy :", accuracy_nb)

Confusion Matrix :
[[965   1]
 [ 14 135]]

Classification Report :
              precision    recall  f1-score   support

           0       0.99      1.00      0.99       966
           1       0.99      0.91      0.95       149

    accuracy                           0.99      1115
   macro avg       0.99      0.95      0.97      1115
weighted avg       0.99      0.99      0.99      1115

Train accuracy : 0.9928202827013687
Test accuracy : 0.9865470852017937


**Test important a realiser en plus de l'accuracy sur le **model LogisticRegression** **

In [54]:
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import accuracy_score

print("Confusion Matrix :")
print(confusion_matrix(y_test, y_pred_lr))

print("\nClassification Report :")
print(classification_report(y_test, y_pred_lr))


train_pred = model_lr.predict(X_train)
print("Train accuracy :", accuracy_score(y_train, train_pred))
print("Test accuracy :", accuracy_lr)

Confusion Matrix :
[[962   4]
 [ 11 138]]

Classification Report :
              precision    recall  f1-score   support

           0       0.99      1.00      0.99       966
           1       0.97      0.93      0.95       149

    accuracy                           0.99      1115
   macro avg       0.98      0.96      0.97      1115
weighted avg       0.99      0.99      0.99      1115

Train accuracy : 0.9997756338344178
Test accuracy : 0.9865470852017937


**Model 3** qui compare les deux premier

In [63]:
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score


voting_model = VotingClassifier(
    estimators=[
        ('nb', model_nb),
        ('lr', model_lr)
    ],
    voting='soft'
)

voting_model.fit(X_train, y_train)

y_pred_vote = voting_model.predict(X_test)

print(confusion_matrix(y_test, y_pred_vote))
print(classification_report(y_test, y_pred_vote))

train_pred_vote = voting_model.predict(X_train)

print("Train accuracy :", accuracy_score(y_train, train_pred_vote))
print("Test accuracy :", accuracy_score(y_test, y_pred_vote))


[[965   1]
 [ 12 137]]
              precision    recall  f1-score   support

           0       0.99      1.00      0.99       966
           1       0.99      0.92      0.95       149

    accuracy                           0.99      1115
   macro avg       0.99      0.96      0.97      1115
weighted avg       0.99      0.99      0.99      1115

Train accuracy : 0.9988781691720888
Test accuracy : 0.9883408071748879


# Remarque

#### dataset est fortement déséquilibré :

* (87%) des messages sont des messages normaux (ham), contre (13%) pour les spam.
* Desequilibre coherant a la realiter et au moyenne oberver.

#### travaille sur le data set :

* les text pouvant contenir **des majuscule, de la ponctuation** ... cella pourrait ralentir la detection, nous effectuons donc un clean du message

* l'ia ne pouvant pas comprendre les mots, **les message sont donc vectoriser** affin de mesurer l’importance des mots dans chaque SMS

* de plus les label symplifier en 0/1 **HAM = 0 et Spam = 1**

* il est necessaire de **separer le data set en 2** affin d'en avoir un pour l'**entrainement et un pour les test** affin d'eviter d'effectuer des test biesier. attention comme le set de données nest pas egalitaire il est necesaire d'utiliser l'option stratify dans train_test_split().

#### Info Supplementaire necessaire et metrique interessante:

* une comparaison entre train accuracy et test accuracy affin de verifier si le model apprend bien. si les deux variable sont proche alors le model apres bien si il y a un trop grand écare, le model memorise et n'apprend pas vraiment. attention au surapprentisage.7

1. Recall (spam) => Le recall est une mesure de performance couramment utilisée pour les modèles de classification.
2. Precision (spam) => éviter faux positifs
3. F1-score => équilibre des deux

##### Model Multinomial Naive Bayes :
1. Il calcule quelle est la probabilité que ce message soit un spam, étant donné les mots qu’il contient ?
2. Concrètement :
* Il compte la fréquence des mots dans spam vs ham
* Il compare les probabilités
3. Avantages :
* Très rapide
* Très efficace pour le texte
* Peu de paramètres


##### Model Logistic Regression :
1. Logistic Regression est un modèle linéaire qui combine les mots avec des poids pour prédire la probabilité qu’un message soit un spam.
2. Concrètement :
* Chaque mot influence la décision
* Le modèle apprend automatiquement les poids
3. Avantages :
* Plus flexible que Naive Bayes
* Meilleur équilibre precision/recall
* Pas d’hypothèse d’indépendance

##### Model Voting Classifier
1. Combiner plusieurs modèles pour améliorer les performances
2. Concrètement :
* Utilise plusieurs modèles
* Combine leurs prédictions
3. Avantages :
* Plus robuste
* Réduit les erreurs
*   Exploite les forces de chaque modèle

# resultat

### test 1 :  model 1 Multinomial Naive Bayes (MultinomialNB())

Confusion Matrix :

|           | Prédit HAM | Prédit SPAM |
| --------- | ---------- | ----------- |
| Vrai HAM  | 966        | 0           |
| Vrai SPAM | 44         | 105         |

Classification Report :

|              | precision | recall | f1-score |support |
|--------------|-----------|--------|----------|--------|
| HAM  (0)     | 0.96      | 1.00   | 0.98     |966     |
| SPAM (1)     | 1.00      | 0.70   | 0.83     |149     |
| Accuracy     |           |        | 0.96     |1115    |
| macro avg    | 0.98      | 0.85   | 0.90     |1115    |
| weighted avg | 0.96      | 0.96   | 0.96     |1115    |


conclusion :
 Attention ce nest pas parce que jai un accuracy elever (~0.96) que mon model est precis. les spam etant moins presant dans le data set cette données nest pas la plus pertinante.
 Ne fait AUCUNE erreur sur les HAM (0 faux positifs) par contre il a rater 44 spam sur 155 tester soit -1/3. cest bien de ne pas classer des vrais mail en spam mais il peut etre dangereux de laisser passer auttant de spam.


### test 2 :  model 1 Multinomial Naive Bayes (MultinomialNB(alpha=0.5))
Confusion Matrix :

|           | Prédit HAM | Prédit SPAM |
| --------- | ---------- | ----------- |
| Vrai HAM  | 965        | 1           |
| Vrai SPAM | 26         | 123         |

Classification Report :

|              | precision | recall | f1-score |support |
|--------------|-----------|--------|----------|--------|
| HAM  (0)     | 0.97      | 1.00   | 0.99     |966     |
| SPAM (1)     | 0.99      | 0.83   | 0.90     |149     |
| Accuracy     |           |        | 0.98     |1115    |
| macro avg    | 0.98      | 0.91   | 0.94     |1115    |
| weighted avg | 0.98      | 0.98   | 0.97     |1115    |

Train accuracy : 0.988332959389724
Test accuracy : 0.9757847533632287

conlusion:
En réduisant le paramètre alpha, le modèle devient plus sensible et améliore significativement le recall sur les spams (de 0.70 à 0.83), au prix d’une très légère augmentation des faux positifs.
ajoue de la comparaison de train accuracy et test accuracy affin de verifier si le model apprend bien.

### test 3 model 1 Multinomial Naive Bayes (MultinomialNB(alpha=0.05))

Confusion Matrix :

|           | Prédit HAM | Prédit SPAM |
| --------- |------------|-------------|
| Vrai HAM  | 959        | 7           |
| Vrai SPAM | 9          | 140         |

Classification Report :

|              | precision | recall | f1-score |support |
|--------------|-----------|--------|----------|--------|
| HAM  (0)     | 0.99      | 0.99   | 0.99     |966     |
| SPAM (1)     | 0.95      | 0.94   | 0.95     |149     |
| Accuracy     |           |        | 0.99     |1115    |
| macro avg    | 0.97      | 0.97   | 0.97     |1115    |
| weighted avg | 0.99      | 0.99   | 0.99     |1115    |

Train accuracy : 0.9982050706753421
Test accuracy : 0.9856502242152466

Conclusin:

~ 6% des spam envoyer sont considere comme des ham pour seulement  0.7% de Ham qui sont detecter comme Spam

### test 4 :  model 1 Multinomial Naive Bayes (MultinomialNB(alpha=0.05)) + TfidfVectorizer(ngram_range=(1,2)) lors de la vectorisation


Confusion Matrix :

|           | Prédit HAM | Prédit SPAM |
| --------- |------------|-------------|
| Vrai HAM  | 931        | 35          |
| Vrai SPAM | 4          | 145         |

Classification Report :

|              | precision | recall | f1-score |support |
|--------------|-----------|--------|----------|--------|
| HAM  (0)     | 1.00      | 0.96   | 0.98     |966     |
| SPAM (1)     | 0.81      | 0.97   | 0.88     |149     |
| Accuracy     |           |        | 0.97     |1115    |
| macro avg    | 0.90      | 0.97   | 0.93     |1115    |
| weighted avg | 0.97      | 0.97   | 0.97     |1115    |

Train accuracy : 0.9997756338344178
Test accuracy : 0.9650224215246637

conclusion :

~2.6% de spam passe encore par contre 3.6% de Ham sont considere comme spam. Attention le train accuracy et test accuracy commence a diverger attention au sur apprentisage. le test 3 est plus viable mais, test 3 expérience utilisateu vs test 4 sécurité maximale.

### test 5 : model 2 Logistic Regression: model_lr = LogisticRegression(max_iter=1000)

Confusion Matrix :

|           | Prédit HAM | Prédit SPAM |
| --------- |------------|-------------|
| Vrai HAM  | 966        | 0           |
| Vrai SPAM | 58         | 91          |

Classification Report :

|              | precision | recall | f1-score |support |
|--------------|-----------|--------|----------|--------|
| HAM  (0)     | 0.94      | 1.00   | 0.97     |966     |
| SPAM (1)     | 1.00      | 0.61   | 0.76     |149     |
| Accuracy     |           |        | 0.95     |1115    |
| macro avg    | 0.97      | 0.81   | 0.86     |1115    |
| weighted avg | 0.95      | 0.95   | 0.94     |1115    |

Train accuracy : 0.9997756338344178
Test accuracy : 0.9479820627802691

conclusion : plus de 1/3 des spam sont detecter en Ham ce qui est un resultat moins bon que le model precedans lors du premiere essaye, de plus le train accuracy est largement supperieur au test accuracy se qui laisse sous entendre un sur apprentisage.

### test 6 : Model 2 Logistic Regression : model_lr = LogisticRegression(C=1, class_weight='balanced', max_iter=2000)

Confusion Matrix :

|           | Prédit HAM | Prédit SPAM |
| --------- |------------|-------------|
| Vrai HAM  | 963        | 3           |
| Vrai SPAM | 21         | 128         |


Classification Report :

|              | precision | recall | f1-score |support |
|--------------|-----------|--------|----------|--------|
| HAM  (0)     | 0.98      | 1.00   | 0.99     |966     |
| SPAM (1)     | 0.98      | 0.86   | 0.91     |149     |
| Accuracy     |           |        | 0.98     |1115    |
| macro avg    | 0.98      | 0.93   | 0.95     |1115    |
| weighted avg | 0.98      | 0.98   | 0.98     |1115    |

Train accuracy : 0.9997756338344178
Test accuracy : 0.97847533632287

conlcution: le model a largement augmenter cest capaciter. il detect encore 13% des Spam en Ham, neaimoins le Train accuracy se rapproche du Test accuracy se qui est bon signe. <br>
test effectuer avec C = 0.1 | C = 1 | C = 10 | C = 100  => meme resultat <br>
test effectuer avec max_iter = 2000 | max_iter = 1000 => meme resultat <br>
test effectuer avec solver='liblinear' => plus de Ham etait detecter en Spam que lorque solver nest pas definie <br>

### Test 7 : Model 2  Logistic Regression : Meme model que le test precedant mais avec TfidfVectorizer(ngram_range=(1,2), max_features=5000) (vectorisation different)

Confusion Matrix :

|           | Prédit HAM | Prédit SPAM |
| --------- |------------|-------------|
| Vrai HAM  | 961        | 4           |
| Vrai SPAM | 11         | 138         |

Classification Report :

|              | precision | recall | f1-score |support |
|--------------|-----------|-------|----------|--------|
| HAM  (0)     | 0.99      | 0.99  | 0.99     |966     |
| SPAM (1)     | 0.97      | 0.93  | 0.95     |149     |
| Accuracy     |           |       | 0.99     |1115    |
| macro avg    | 0.98      | 0.96  | 0.97     |1115    |
| weighted avg | 0.99      | 0.99  | 0.99     |1115    |

Train accuracy : 0.9928202827013687
Test accuracy : 0.9856502242152466


conlcution: le model detect plus que 7% de Spam en Ham, et sont Train accuracy est tres proche du Test accuracy ce qui est tres bon signe. les resultat sont comparable a ceux obtenue lors du test 3 avec le model 1.

### Test 8 : Model 3  VotingClassifier : le model compare les deux resultat des model nb et lr. Hard voting chaqu'un un vote

Confusion Matrix :

|           | Prédit HAM | Prédit SPAM |
| --------- |------------|-------------|
| Vrai HAM  | 965        | 1           |
| Vrai SPAM | 15         | 134         |

Classification Report :

|              | precision | recall | f1-score |support |
|--------------|-----------|--------|----------|--------|
| HAM  (0)     | 0.98      | 1.00   | 0.99     |966     |
| SPAM (1)     | 0.99      | 0.90   | 0.94     |149     |
| Accuracy     |           |        | 0.99     |1115    |
| macro avg    | 0.98      | 0.95   | 0.97     |1115    |
| weighted avg | 0.99      | 0.99   | 0.99     |1115    |

Train accuracy : 0.9932690150325331
Test accuracy : 0.9856502242152466

conlcution :  le resultat de se model est moins performant que les deux model individuelement (test 3 + test 7). pour l'intant le model nest pas pertinent

### Test 9 : Model 3  VotingClassifier : le model compare les deux resultat des model nb et lr. soft voting chaqu'un dit a comb il pense que cest un spam

Confusion Matrix :

|           | Prédit HAM | Prédit SPAM |
| --------- |------------|-------------|
| Vrai HAM  | 965        | 1           |
| Vrai SPAM | 12         | 137         |

Classification Report :

|              | precision | recall | f1-score |support |
|--------------|-----------|--------|----------|--------|
| HAM  (0)     | 0.99      | 1.00   | 0.99     |966     |
| SPAM (1)     | 0.99      | 0.92   | 0.95     |149     |
| Accuracy     |           |        | 0.99     |1115    |
| macro avg    | 0.98      | 0.96   | 0.97     |1115    |
| weighted avg | 0.99      | 0.99   | 0.99     |1115    |

Train accuracy : 0.9988781691720888
Test accuracy : 0.9883408071748879

Conclusion: Le model a un principe interessant et peut etre plus performant avec plus de model different. les resultat reste neamoins tres concluant.


# Conclusion :
Dans ce projet, nous avons cherché à classifier des messages SMS en spam ou non-spam à partir de leur contenu textuel.

Après une phase de preprocessing incluant le nettoyage des données et la vectorisation via TF-IDF, plusieurs modèles ont été testés : Multinomial Naive Bayes, Logistic Regression et une approche par ensemble (Voting).

Les modèles individuels ont montré d’excellentes performances, avec une accuracy proche de 99%. Logistic Regression a permis d’obtenir un meilleur recall, tandis que Naive Bayes offrait une précision légèrement supérieure.

Afin de tirer parti des différences entre ces modèles, une approche de type Voting a été mise en place. Le voting “hard” n’a pas apporté d’amélioration, mais le voting “soft”, basé sur les probabilités, a permis d’obtenir un modèle plus équilibré.

Le modèle final atteint une accuracy de 98.8%, avec un recall de 92% sur les spams et une précision de 99%, ce qui en fait un modèle fiable et performant.


**J’ai testé plusieurs modèles pour détecter les spams SMS. Naive Bayes et Logistic Regression donnent tous les deux d’excellents résultats. Logistic Regression détecte un peu mieux les spams, tandis que Naive Bayes fait moins d’erreurs sur les messages normaux.
J’ai ensuite combiné les deux avec un VotingClassifier. Le soft voting donne le meilleur équilibre avec 98.8% d’accuracy et 92% de recall sur les spams.
Le modèle est donc fiable et généralise bien.**